In [5]:
# 학습된 모델 대신 직접 생성한 embedding을 사용(toy-embedding)
import math
import re
import pandas as pd
import torch
import torch.nn.functional as F

In [6]:
sentences = [
    '이 영화 정말 좋다',
    '이 영화 너무 지루하다',
    '배우 연기가 감동적이다',
    '전개는 보통이지만 마지막이 재미있다'
]

In [7]:
# toy-embedding : 2차원 벡터를 임의로 생성
# positive단어는 오른쪽 위 쪽에 배치
# negative 단어는 왼쪽 아래쪽에 배치
# neutal 단어는 원점 근처에 배치

# query와 비슷한 방향의 단어가 더 큰 score를 받음
toy_vocab = {
    '이': torch.tensor([0.0, 0.0]),
    '영화': torch.tensor([0.1, 0.0]),
    '정말': torch.tensor([0.0, 0.2]),
    '좋다': torch.tensor([1.2, 1.0]),
    '너무': torch.tensor([0.0, 0.1]),
    '지루하다': torch.tensor([-1.2, -1.0]),
    '배우': torch.tensor([0.2, 0.1]),
    '연기가': torch.tensor([0.3, 0.1]),
    '감동적이다': torch.tensor([1.0, 1.2]),
    '전개는': torch.tensor([0.0, 0.0]),
    '보통이지만': torch.tensor([0.0, 0.0]),
    '마지막이': torch.tensor([0.0, 0.1]),
    '재미있다': torch.tensor([1.1, 1.1]),
    '별로다': torch.tensor([-1.0, -1.1]),
    '최악이다': torch.tensor([-1.3, -1.2]),
}

poistive_query = torch.tensor([1.0,1.0])
negative_query = torch.tensor([-1.0,-1.0])
toy_vocab['좋다'], poistive_query, negative_query

(tensor([1.2000, 1.0000]), tensor([1., 1.]), tensor([-1., -1.]))

## 전처리 함수 만들기
- 문장을 토큰으로나눈뒤, 각 토큰을 toy embedding 벡터로 바꾼다
- 여기서 key value는 같은 벡터를 사용, query는 지금 찾고싶은 의미

In [8]:
def simple_tokenize(text: str):
    text = re.sub(r'[^가-힣\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return []
    return text.split()

def get_token_vectors(tokens, vocab):
    vectors = []
    for token in tokens:
        vectors.append(vocab.get(token, torch.tensor([0.0, 0.0])))
    return torch.stack(vectors) if vectors else torch.empty(0, 2)

def attention_with_query(tokens, query, vocab):
    key = get_token_vectors(tokens, vocab)
    value = key

    if key.numel() == 0:
        return pd.DataFrame(), torch.tensor([])

    d_k = key.size(-1)
    scores = torch.matmul(key, query) / math.sqrt(d_k)
    weights = F.softmax(scores, dim=0)
    context = torch.sum(weights.unsqueeze(-1) * value, dim=0)

    frame = pd.DataFrame({
        'token': tokens,
        'key_x': key[:, 0].tolist(),
        'key_y': key[:, 1].tolist(),
        'score': scores.tolist(),
        'weight': weights.tolist(),
    })
    frame['value_x'] = frame['key_x']
    frame['value_y'] = frame['key_y']
    return frame, context

## 한문장에 query/ key /value 
- 첫번째 문장에 positive query를 넣어서 어떤 토큰이 더 중요한지 확인

In [10]:
toy_vocab

{'이': tensor([0., 0.]),
 '영화': tensor([0.1000, 0.0000]),
 '정말': tensor([0.0000, 0.2000]),
 '좋다': tensor([1.2000, 1.0000]),
 '너무': tensor([0.0000, 0.1000]),
 '지루하다': tensor([-1.2000, -1.0000]),
 '배우': tensor([0.2000, 0.1000]),
 '연기가': tensor([0.3000, 0.1000]),
 '감동적이다': tensor([1.0000, 1.2000]),
 '전개는': tensor([0., 0.]),
 '보통이지만': tensor([0., 0.]),
 '마지막이': tensor([0.0000, 0.1000]),
 '재미있다': tensor([1.1000, 1.1000]),
 '별로다': tensor([-1.0000, -1.1000]),
 '최악이다': tensor([-1.3000, -1.2000])}

In [11]:
sentence = sentences[0]
tokens = simple_tokenize(sentence)
frame,context = attention_with_query(tokens,poistive_query,toy_vocab)
print('sentence:', sentence)
print('tokens:', tokens)
print('query:',poistive_query.tolist())
print('context:',context.tolist())
display(frame.sort_values('weight', ascending=False).reset_index(drop=True))

sentence: 이 영화 정말 좋다
tokens: ['이', '영화', '정말', '좋다']
query: [1.0, 1.0]
context: [0.7274695634841919, 0.6239237189292908]


,token,key_x,key_y,score,weight,value_x,value_y
0,좋다,1.2,1.0,1.555635,0.594993,1.2,1.0
1,정말,0.0,0.2,0.141421,0.144653,0.0,0.2
2,영화,0.1,0.0,0.070711,0.134778,0.1,0.0
3,이,0.0,0.0,0.000000,0.125576,0.0,0.0


- 의미해석 : 각 토큰을 toy-vector로 바뀐뒤 query와 의 유사도를 계산
- weight 는 score에 softmax를 적용한 확률기반의 점수
- 모델은 이문장에서 좋다 라는 단어를 중요하게 본다
- context: [0.7274695634841919, 0.6239237189292908]는 이 가중치를 반영해 만든 문장요약 벡터
- 결국 attention은 문장 전테를 다 똑같이 보지 않고 지금 query와 가장 관련있는 단어에 집중한다 

```
query = 지금 찾고 싶은 감정 방향
key = 각 단어가 가진 특징
score = query와 key가 얼마나 비슷한가
weight = score를 확률처럼 바꾼 값
context = weight로 단어들을 섞어 만든 최종 요약
```

## 같은 문장을 positive query와 negative query로 비교

query가 바뀌면 attention weight도 바뀐다.

- positive query: 좋은 단어 쪽을 더 본다.
- negative query: 나쁜 단어 쪽을 더 본다.

In [ ]:
compare_sentence = '이 영화 정말 좋다 하지만 마지막이 별로다'
tokens = simple_tokenize(compare_sentence)
pos_trame, pos_context = attention_with_query(tokens,poistive_query,toy_vocab)
neg_trame, neg_context = attention_with_query(tokens,negative_query,toy_vocab)

print('sentence:', sentence)
print('tokens:', tokens)
print('query:',poistive_query.tolist())
print('context:',pos_context.tolist())
display(pos_trame.sort_values('weight', ascending=False).reset_index(drop=True))

print('sentence:', sentence)
print('tokens:', tokens)
print('query:',negative_query.tolist())
print('context:',neg_context.tolist())
display(neg_trame.sort_values('weight', ascending=False).reset_index(drop=True))

['이', '영화', '정말', '좋다', '하지만', '마지막이', '별로다']